# Case Study 4: Binary Text Classification — Spam SMS

## RNN vs LSTM vs GRU Comprehensive Comparison

---

### Objective
Classify SMS messages as **ham** (legitimate) or **spam** using three recurrent architectures:
- **Vanilla RNN** — simple recurrence, prone to vanishing gradients
- **LSTM** — gated memory cells (forget, input, output gates)
- **GRU** — simplified gating (reset, update gates)

### What You Will Learn
1. How to preprocess raw text data for sequence classification
2. Building vocabularies and word-to-index mappings from scratch
3. Handling class imbalance with weighted loss functions
4. Building and comparing RNN, LSTM, GRU text classifiers in PyTorch
5. Evaluation with F1, AUC-ROC, Precision-Recall curves
6. Hyperparameter tuning with systematic search
7. Comparison against a TF-IDF + Logistic Regression baseline

### Dataset
- **SMS Spam Collection**: 5,574 messages (87% ham / 13% spam)
- Tab-separated file with two columns: label and message text
- Significant class imbalance — requires careful metric selection

---
## 1. Environment Setup

In [ ]:
import random
import time
import string
import re
import warnings
warnings.filterwarnings('ignore')
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, confusion_matrix, classification_report
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {DEVICE}')

# Plot style
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
COLORS = {'RNN': '#e74c3c', 'LSTM': '#2ecc71', 'GRU': '#3498db'}

---
## 2. Data Loading & Exploratory Data Analysis

In [ ]:
# Load dataset
df = pd.read_csv(
    'Text_and_NLP/spam_sms/SMSSpamCollection',
    sep='\t',
    header=None,
    names=['label', 'message']
)
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'\nNull values:\n{df.isnull().sum()}')
print(f'\nDuplicates: {df.duplicated().sum()}')
df.head(10)

In [ ]:
# Class distribution
class_counts = df['label'].value_counts()
class_pcts = df['label'].value_counts(normalize=True) * 100

print('Class Distribution:')
print(f'  ham:  {class_counts["ham"]:,} ({class_pcts["ham"]:.1f}%)')
print(f'  spam: {class_counts["spam"]:,} ({class_pcts["spam"]:.1f}%)')
print(f'  Imbalance ratio: {class_counts["ham"] / class_counts["spam"]:.1f}:1')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
bars = axes[0].bar(
    class_counts.index, class_counts.values,
    color=['#2ecc71', '#e74c3c'], edgecolor='white', width=0.5
)
for bar, val, pct in zip(bars, class_counts.values, class_pcts.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
        f'{val:,} ({pct:.1f}%)', ha='center', fontweight='bold', fontsize=12
    )
axes[0].set_title('Class Distribution', fontsize=14)
axes[0].set_ylabel('Count')
axes[0].set_xlabel('Label')

# Pie chart
axes[1].pie(
    class_counts.values, labels=class_counts.index,
    autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
    startangle=90, textprops={'fontsize': 13}
)
axes[1].set_title('Class Proportion', fontsize=14)

plt.suptitle('SMS Spam Collection — Class Imbalance', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# Feature engineering for EDA
df['msg_length'] = df['message'].apply(len)
df['word_count'] = df['message'].apply(lambda x: len(x.split()))

print('Message Length Statistics (characters):')
print(df.groupby('label')['msg_length'].describe().round(1))
print(f'\nWord Count Statistics:')
print(df.groupby('label')['word_count'].describe().round(1))

In [ ]:
# Message length distribution by class
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Character length
for label, color in [('ham', '#2ecc71'), ('spam', '#e74c3c')]:
    subset = df[df['label'] == label]
    axes[0].hist(
        subset['msg_length'], bins=50, alpha=0.6,
        color=color, label=f'{label} (n={len(subset)})', edgecolor='white'
    )
axes[0].set_title('Message Length Distribution (Characters)', fontsize=13)
axes[0].set_xlabel('Number of Characters')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=11)
axes[0].axvline(
    df[df['label'] == 'ham']['msg_length'].mean(),
    color='#27ae60', linestyle='--', linewidth=2, label='ham mean'
)
axes[0].axvline(
    df[df['label'] == 'spam']['msg_length'].mean(),
    color='#c0392b', linestyle='--', linewidth=2, label='spam mean'
)
axes[0].legend(fontsize=10)

# Word count
for label, color in [('ham', '#2ecc71'), ('spam', '#e74c3c')]:
    subset = df[df['label'] == label]
    axes[1].hist(
        subset['word_count'], bins=40, alpha=0.6,
        color=color, label=f'{label} (n={len(subset)})', edgecolor='white'
    )
axes[1].set_title('Word Count Distribution', fontsize=13)
axes[1].set_xlabel('Number of Words')
axes[1].set_ylabel('Count')
axes[1].legend(fontsize=11)

plt.suptitle('Spam Messages Tend to Be Longer', fontsize=14)
plt.tight_layout()
plt.show()

print('Key observation: Spam messages are significantly longer on average.')
print(f'  Ham mean length:  {df[df["label"]=="ham"]["msg_length"].mean():.0f} chars, '
      f'{df[df["label"]=="ham"]["word_count"].mean():.0f} words')
print(f'  Spam mean length: {df[df["label"]=="spam"]["msg_length"].mean():.0f} chars, '
      f'{df[df["label"]=="spam"]["word_count"].mean():.0f} words')

In [ ]:
# Box plots for message length by class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(
    data=df, x='label', y='msg_length', ax=axes[0],
    palette={'ham': '#2ecc71', 'spam': '#e74c3c'}
)
axes[0].set_title('Character Length by Class', fontsize=13)
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Message Length (chars)')

sns.boxplot(
    data=df, x='label', y='word_count', ax=axes[1],
    palette={'ham': '#2ecc71', 'spam': '#e74c3c'}
)
axes[1].set_title('Word Count by Class', fontsize=13)
axes[1].set_xlabel('Label')
axes[1].set_ylabel('Word Count')

plt.tight_layout()
plt.show()

In [ ]:
# Top 20 most frequent words per class
def get_top_words(messages, n=20):
    """Get top n most frequent words from a list of messages."""
    all_words = []
    for msg in messages:
        words = msg.lower().translate(str.maketrans('', '', string.punctuation)).split()
        all_words.extend(words)
    return Counter(all_words).most_common(n)

ham_top = get_top_words(df[df['label'] == 'ham']['message'], 20)
spam_top = get_top_words(df[df['label'] == 'spam']['message'], 20)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Ham top words
words_h, counts_h = zip(*ham_top)
axes[0].barh(range(len(words_h)), counts_h, color='#2ecc71', edgecolor='white')
axes[0].set_yticks(range(len(words_h)))
axes[0].set_yticklabels(words_h, fontsize=10)
axes[0].invert_yaxis()
axes[0].set_title('Top 20 Words in HAM Messages', fontsize=13)
axes[0].set_xlabel('Frequency')

# Spam top words
words_s, counts_s = zip(*spam_top)
axes[1].barh(range(len(words_s)), counts_s, color='#e74c3c', edgecolor='white')
axes[1].set_yticks(range(len(words_s)))
axes[1].set_yticklabels(words_s, fontsize=10)
axes[1].invert_yaxis()
axes[1].set_title('Top 20 Words in SPAM Messages', fontsize=13)
axes[1].set_xlabel('Frequency')

plt.suptitle('Most Frequent Words by Class', fontsize=14)
plt.tight_layout()
plt.show()

print('Observation: Spam messages contain more promotional words like "free", "call", "txt", "claim"')
print('Ham messages have everyday conversational words like "i", "you", "ok", "go"')

In [ ]:
# Sample messages
print('='*80)
print('SAMPLE HAM MESSAGES')
print('='*80)
for i, msg in enumerate(df[df['label'] == 'ham']['message'].sample(5, random_state=SEED).values):
    print(f'  [{i+1}] {msg[:120]}...' if len(msg) > 120 else f'  [{i+1}] {msg}')

print(f'\n{"="*80}')
print('SAMPLE SPAM MESSAGES')
print('='*80)
for i, msg in enumerate(df[df['label'] == 'spam']['message'].sample(5, random_state=SEED).values):
    print(f'  [{i+1}] {msg[:120]}...' if len(msg) > 120 else f'  [{i+1}] {msg}')

---
## 3. Text Preprocessing & Vocabulary Building

In [ ]:
# Text preprocessing function
def preprocess_text(text):
    """Lowercase, remove punctuation, split into tokens."""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', ' NUM ', text)  # Replace numbers with NUM token
    tokens = text.split()
    return tokens

# Apply preprocessing
df['tokens'] = df['message'].apply(preprocess_text)
df['num_tokens'] = df['tokens'].apply(len)

# Encode labels
df['label_enc'] = (df['label'] == 'spam').astype(int)

print('Preprocessing examples:')
for i in range(3):
    print(f'  Original: {df["message"].iloc[i][:80]}...')
    print(f'  Tokens:   {df["tokens"].iloc[i][:10]}...')
    print(f'  Label:    {df["label"].iloc[i]} -> {df["label_enc"].iloc[i]}')
    print()

In [ ]:
# Stratified train/val/test split (70/15/15)
X_all = df['tokens'].values
y_all = df['label_enc'].values

# First split: 70% train, 30% temp
X_train_tokens, X_temp_tokens, y_train, y_temp = train_test_split(
    X_all, y_all, test_size=0.30, random_state=SEED, stratify=y_all
)

# Second split: 50/50 of temp -> 15% val, 15% test
X_val_tokens, X_test_tokens, y_val, y_test = train_test_split(
    X_temp_tokens, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f'Train: {len(X_train_tokens)} ({len(X_train_tokens)/len(X_all)*100:.1f}%)')
print(f'Val:   {len(X_val_tokens)} ({len(X_val_tokens)/len(X_all)*100:.1f}%)')
print(f'Test:  {len(X_test_tokens)} ({len(X_test_tokens)/len(X_all)*100:.1f}%)')

print(f'\nClass distribution in each split:')
for name, y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    spam_pct = y.mean() * 100
    print(f'  {name}: {spam_pct:.1f}% spam, {100-spam_pct:.1f}% ham')

In [ ]:
# Build vocabulary from training set ONLY
PAD_IDX = 0
UNK_IDX = 1
MIN_FREQ = 2  # Minimum word frequency to include

word_counts = Counter()
for tokens in X_train_tokens:
    word_counts.update(tokens)

# Build word-to-index mapping
word2idx = {'<PAD>': PAD_IDX, '<UNK>': UNK_IDX}
idx = 2
for word, count in word_counts.most_common():
    if count >= MIN_FREQ:
        word2idx[word] = idx
        idx += 1

VOCAB_SIZE = len(word2idx)

print(f'Total unique words in training set: {len(word_counts):,}')
print(f'Words with freq >= {MIN_FREQ}: {VOCAB_SIZE - 2:,}')
print(f'Vocabulary size (with PAD, UNK): {VOCAB_SIZE:,}')
print(f'\nMost common words (top 15):')
for word, count in word_counts.most_common(15):
    print(f'  {word:15s} -> idx={word2idx[word]:5d}  (freq={count})')

In [ ]:
# Sequence padding/truncation
MAX_SEQ_LEN = 50  # Based on 95th percentile of word counts

pct_95 = np.percentile(df['num_tokens'].values, 95)
pct_99 = np.percentile(df['num_tokens'].values, 99)
print(f'Token count percentiles:')
print(f'  50th: {np.percentile(df["num_tokens"].values, 50):.0f}')
print(f'  75th: {np.percentile(df["num_tokens"].values, 75):.0f}')
print(f'  90th: {np.percentile(df["num_tokens"].values, 90):.0f}')
print(f'  95th: {pct_95:.0f}')
print(f'  99th: {pct_99:.0f}')
print(f'\nUsing MAX_SEQ_LEN = {MAX_SEQ_LEN}')
coverage = (df['num_tokens'] <= MAX_SEQ_LEN).mean() * 100
print(f'Coverage: {coverage:.1f}% of messages fit without truncation')


def tokens_to_indices(tokens, word2idx, max_len):
    """Convert token list to padded/truncated index sequence."""
    indices = [word2idx.get(tok, UNK_IDX) for tok in tokens]
    # Truncate
    if len(indices) > max_len:
        indices = indices[:max_len]
    # Pad
    while len(indices) < max_len:
        indices.append(PAD_IDX)
    return indices


# Convert all splits to index sequences
X_train_idx = np.array([tokens_to_indices(t, word2idx, MAX_SEQ_LEN) for t in X_train_tokens])
X_val_idx = np.array([tokens_to_indices(t, word2idx, MAX_SEQ_LEN) for t in X_val_tokens])
X_test_idx = np.array([tokens_to_indices(t, word2idx, MAX_SEQ_LEN) for t in X_test_tokens])

print(f'\nEncoded shapes:')
print(f'  X_train: {X_train_idx.shape}')
print(f'  X_val:   {X_val_idx.shape}')
print(f'  X_test:  {X_test_idx.shape}')

In [ ]:
# Class weight computation for imbalance
n_ham = (y_train == 0).sum()
n_spam = (y_train == 1).sum()
pos_weight = torch.tensor([n_ham / n_spam], dtype=torch.float32)

print(f'Training set: {n_ham} ham, {n_spam} spam')
print(f'pos_weight for BCEWithLogitsLoss: {pos_weight.item():.2f}')
print(f'This upweights the spam (positive) class by {pos_weight.item():.1f}x')

In [ ]:
# Custom Dataset and DataLoaders
class TextDataset(Dataset):
    """Custom dataset for text classification."""
    def __init__(self, X, y):
        self.X = torch.LongTensor(X)
        self.y = torch.FloatTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


BATCH_SIZE = 64

train_dataset = TextDataset(X_train_idx, y_train)
val_dataset = TextDataset(X_val_idx, y_val)
test_dataset = TextDataset(X_test_idx, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Verify shapes
X_batch, y_batch = next(iter(train_loader))
print(f'Batch shapes:')
print(f'  X: {X_batch.shape}  (batch_size, seq_len)')
print(f'  y: {y_batch.shape}  (batch_size,)')
print(f'\nDataLoader sizes:')
print(f'  Train: {len(train_loader)} batches')
print(f'  Val:   {len(val_loader)} batches')
print(f'  Test:  {len(test_loader)} batches')

---
## 4. Model Architecture

### Architecture Diagram
```
Input: (batch, seq_len=50)           [token indices]
       |
  +-----------+
  | Embedding |  (vocab_size, embed_dim)  -> (batch, seq_len, embed_dim)
  +-----------+
       |
  +--------+   +--------+   +--------+         +--------+
  | RNN /  |-->| RNN /  |-->| RNN /  |--> ... ->| RNN /  |--> h_T
  | LSTM / |   | LSTM / |   | LSTM / |         | LSTM / |
  | GRU    |   | GRU    |   | GRU    |         | GRU    |
  +--------+   +--------+   +--------+         +--------+
    t=1          t=2          t=3                 t=50
                                                   |
                                              [Dropout]
                                                   |
                                              [Linear Layer]
                                                   |
                                              Output: (batch, 1)
                                              [logit -> Sigmoid -> P(spam)]
```

**Key differences from time series model (NB1):**
- Uses `nn.Embedding` layer to convert token indices to dense vectors
- `padding_idx=0` tells embedding to output zeros for PAD tokens
- Output is a single logit (binary classification)
- Uses `BCEWithLogitsLoss` with `pos_weight` for class imbalance
- Optional bidirectional mode concatenates forward + backward hidden states

In [ ]:
class TextClassifier(nn.Module):
    """Unified RNN/LSTM/GRU model for binary text classification."""

    SUPPORTED_TYPES = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}

    def __init__(self, model_type, vocab_size, embed_dim, hidden_size,
                 num_layers=1, dropout=0.0, bidirectional=False, pad_idx=0):
        super().__init__()
        assert model_type in self.SUPPORTED_TYPES, \
            f'model_type must be one of {list(self.SUPPORTED_TYPES.keys())}'

        self.model_type = model_type
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional

        # Embedding layer
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )

        # Recurrent layer
        rnn_cls = self.SUPPORTED_TYPES[model_type]
        self.rnn = rnn_cls(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )

        # Dropout before output
        self.dropout = nn.Dropout(dropout)

        # Output layer
        fc_input_size = hidden_size * (2 if bidirectional else 1)
        self.fc = nn.Linear(fc_input_size, 1)

    def forward(self, x):
        # x: (batch, seq_len) — token indices
        embedded = self.embedding(x)          # (batch, seq_len, embed_dim)
        rnn_out, _ = self.rnn(embedded)        # (batch, seq_len, hidden * num_dir)
        last_hidden = rnn_out[:, -1, :]        # (batch, hidden * num_dir)
        dropped = self.dropout(last_hidden)    # (batch, hidden * num_dir)
        logit = self.fc(dropped)               # (batch, 1)
        return logit.squeeze(1)                # (batch,)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [ ]:
# Verify model with a dummy forward pass
for mt in ['RNN', 'LSTM', 'GRU']:
    model = TextClassifier(
        model_type=mt, vocab_size=VOCAB_SIZE, embed_dim=100,
        hidden_size=64, num_layers=1, dropout=0.2, bidirectional=False
    )
    dummy_input = torch.randint(0, VOCAB_SIZE, (4, MAX_SEQ_LEN))
    output = model(dummy_input)
    print(f'{mt:5s}: output shape = {output.shape}, params = {model.count_parameters():,}')

print('\nAll model types produce correct output shape.')

In [ ]:
# Parameter count comparison
param_comparison = []
for model_type in ['RNN', 'LSTM', 'GRU']:
    model = TextClassifier(
        model_type=model_type, vocab_size=VOCAB_SIZE, embed_dim=100,
        hidden_size=64, num_layers=2, dropout=0.2, bidirectional=False
    )
    params = model.count_parameters()
    param_comparison.append({'Model': model_type, 'Parameters': params})
    print(f'{model_type:5s}: {params:,} parameters')

param_df = pd.DataFrame(param_comparison)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    param_df['Model'], param_df['Parameters'],
    color=[COLORS[m] for m in param_df['Model']], edgecolor='white', width=0.5
)
for bar, val in zip(bars, param_df['Parameters']):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
        f'{val:,}', ha='center', fontweight='bold'
    )
ax.set_title(f'Parameter Count Comparison (embed=100, hidden=64, layers=2, vocab={VOCAB_SIZE:,})', fontsize=12)
ax.set_ylabel('Trainable Parameters')
plt.tight_layout()
plt.show()

print(f'\nNote: Most parameters are in the Embedding layer ({VOCAB_SIZE} x 100 = {VOCAB_SIZE*100:,})')
print(f'LSTM has ~4x RNN parameters in the recurrent layer (3 extra gates)')
print(f'GRU has ~3x RNN parameters in the recurrent layer (2 gates vs 0)')

In [ ]:
# Breakdown: Embedding vs Recurrent vs Linear parameters
breakdown_data = []
for model_type in ['RNN', 'LSTM', 'GRU']:
    model = TextClassifier(
        model_type=model_type, vocab_size=VOCAB_SIZE, embed_dim=100,
        hidden_size=64, num_layers=1, dropout=0.0, bidirectional=False
    )
    emb_params = sum(p.numel() for p in model.embedding.parameters())
    rnn_params = sum(p.numel() for p in model.rnn.parameters())
    fc_params = sum(p.numel() for p in model.fc.parameters())
    breakdown_data.append({
        'Model': model_type, 'Embedding': emb_params,
        'Recurrent': rnn_params, 'Linear': fc_params
    })

breakdown_df = pd.DataFrame(breakdown_data).set_index('Model')
print('Parameter Breakdown:')
print(breakdown_df.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
breakdown_df.plot(kind='bar', stacked=True, ax=ax,
                  color=['#95a5a6', '#3498db', '#e74c3c'], edgecolor='white')
ax.set_title('Parameter Breakdown by Layer Type', fontsize=13)
ax.set_ylabel('Parameters')
ax.set_xticklabels(breakdown_df.index, rotation=0)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## 5. Training Infrastructure

In [ ]:
def train_model(model, train_loader, val_loader, epochs, lr,
                pos_weight=None, device=DEVICE, patience=10,
                clip_grad=1.0, verbose=True):
    """Train a binary classifier with early stopping and gradient clipping.

    Returns:
        history: dict with train_loss, val_loss, val_acc, val_f1 lists
        training_time: total training time in seconds
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    pw = pos_weight.to(device) if pos_weight is not None else None
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=5, factor=0.5, verbose=False
    )

    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
    best_val_f1 = 0.0
    best_state = None
    patience_counter = 0

    start_time = time.time()

    for epoch in range(epochs):
        # Training
        model.train()
        train_losses = []
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()
            train_losses.append(loss.item())

        # Validation
        model.eval()
        val_losses = []
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                logits = model(X_batch)
                loss = criterion(logits, y_batch)
                val_losses.append(loss.item())
                preds = (torch.sigmoid(logits) >= 0.5).long()
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y_batch.cpu().numpy())

        avg_train = np.mean(train_losses)
        avg_val = np.mean(val_losses)
        val_acc = accuracy_score(all_labels, all_preds)
        val_f1 = f1_score(all_labels, all_preds)

        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        scheduler.step(avg_val)

        # Early stopping on val_f1
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if verbose:
                    print(f'  Early stopping at epoch {epoch+1}')
                break

        if verbose and (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d}/{epochs} | '
                  f'Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f} | '
                  f'Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}')

    training_time = time.time() - start_time

    # Restore best model
    if best_state is not None:
        model.load_state_dict(best_state)
        model = model.to(device)

    return history, training_time

In [ ]:
def evaluate_classifier(model, data_loader, device=DEVICE):
    """Evaluate a classifier and return metrics + raw outputs.

    Returns:
        metrics: dict with accuracy, f1, precision, recall, auc_roc
        all_probs: numpy array of predicted probabilities
        all_labels: numpy array of true labels
        cm: confusion matrix
    """
    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(y_batch.numpy())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_preds = (all_probs >= 0.5).astype(int)

    metrics = {
        'Accuracy': accuracy_score(all_labels, all_preds),
        'F1': f1_score(all_labels, all_preds),
        'Precision': precision_score(all_labels, all_preds),
        'Recall': recall_score(all_labels, all_preds),
        'AUC-ROC': roc_auc_score(all_labels, all_probs),
    }
    cm = confusion_matrix(all_labels, all_preds)

    return metrics, all_probs, all_labels, cm

In [ ]:
# Quick sanity check — train a small model for 5 epochs
sanity_model = TextClassifier(
    model_type='LSTM', vocab_size=VOCAB_SIZE, embed_dim=50,
    hidden_size=32, num_layers=1, dropout=0.1
)
sanity_history, sanity_time = train_model(
    sanity_model, train_loader, val_loader,
    epochs=5, lr=0.001, pos_weight=pos_weight, verbose=True
)
sanity_metrics, _, _, _ = evaluate_classifier(sanity_model, val_loader)
print(f'\nSanity check metrics: {sanity_metrics}')
print(f'Training pipeline works correctly.')

---
## 6. Hyperparameter Tuning

We perform a systematic search over key hyperparameters for each model type.

**Search space:**
- `hidden_size`: [32, 64, 128]
- `num_layers`: [1, 2]
- `lr`: [0.001, 0.005, 0.01]
- `embedding_dim`: [50, 100]
- `dropout`: [0.0, 0.2]
- `bidirectional`: [True, False]

12 manually selected configurations x 3 model types = **36 total trials**

In [ ]:
# Hyperparameter search configurations (12 configs)
SEARCH_CONFIGS = [
    {'hidden_size': 32,  'num_layers': 1, 'lr': 0.001, 'embed_dim': 50,  'dropout': 0.0, 'bidirectional': False},
    {'hidden_size': 64,  'num_layers': 1, 'lr': 0.001, 'embed_dim': 50,  'dropout': 0.0, 'bidirectional': False},
    {'hidden_size': 64,  'num_layers': 1, 'lr': 0.005, 'embed_dim': 100, 'dropout': 0.0, 'bidirectional': False},
    {'hidden_size': 64,  'num_layers': 2, 'lr': 0.001, 'embed_dim': 100, 'dropout': 0.2, 'bidirectional': False},
    {'hidden_size': 128, 'num_layers': 1, 'lr': 0.001, 'embed_dim': 100, 'dropout': 0.0, 'bidirectional': False},
    {'hidden_size': 128, 'num_layers': 2, 'lr': 0.001, 'embed_dim': 100, 'dropout': 0.2, 'bidirectional': False},
    {'hidden_size': 64,  'num_layers': 1, 'lr': 0.001, 'embed_dim': 50,  'dropout': 0.2, 'bidirectional': True},
    {'hidden_size': 64,  'num_layers': 1, 'lr': 0.005, 'embed_dim': 100, 'dropout': 0.2, 'bidirectional': True},
    {'hidden_size': 128, 'num_layers': 1, 'lr': 0.001, 'embed_dim': 100, 'dropout': 0.2, 'bidirectional': True},
    {'hidden_size': 32,  'num_layers': 1, 'lr': 0.01,  'embed_dim': 50,  'dropout': 0.0, 'bidirectional': False},
    {'hidden_size': 64,  'num_layers': 1, 'lr': 0.01,  'embed_dim': 50,  'dropout': 0.2, 'bidirectional': False},
    {'hidden_size': 128, 'num_layers': 2, 'lr': 0.005, 'embed_dim': 50,  'dropout': 0.2, 'bidirectional': True},
]

TUNING_EPOCHS = 30

print(f'Search space: {len(SEARCH_CONFIGS)} configs x 3 model types = '
      f'{len(SEARCH_CONFIGS) * 3} total trials')
print(f'Tuning epochs per trial: {TUNING_EPOCHS}')

In [ ]:
tuning_results = []

for model_type in ['RNN', 'LSTM', 'GRU']:
    print(f'\n{"="*60}')
    print(f'Tuning {model_type}')
    print(f'{"="*60}')

    for i, config in enumerate(SEARCH_CONFIGS):
        # Set seed for each trial for reproducibility
        torch.manual_seed(SEED)
        np.random.seed(SEED)

        model = TextClassifier(
            model_type=model_type,
            vocab_size=VOCAB_SIZE,
            embed_dim=config['embed_dim'],
            hidden_size=config['hidden_size'],
            num_layers=config['num_layers'],
            dropout=config['dropout'],
            bidirectional=config['bidirectional']
        )

        history, train_time = train_model(
            model, train_loader, val_loader,
            epochs=TUNING_EPOCHS, lr=config['lr'],
            pos_weight=pos_weight, verbose=False
        )

        best_val_f1 = max(history['val_f1'])
        best_val_acc = max(history['val_acc'])
        best_val_loss = min(history['val_loss'])

        tuning_results.append({
            'model_type': model_type,
            'config_id': i,
            **config,
            'best_val_f1': best_val_f1,
            'best_val_acc': best_val_acc,
            'best_val_loss': best_val_loss,
            'train_time': train_time,
            'epochs_run': len(history['val_loss']),
            'params': model.count_parameters()
        })

        bidir_str = 'Bi' if config['bidirectional'] else 'Uni'
        print(f'  Config {i+1:2d}/{len(SEARCH_CONFIGS)}: '
              f'h={config["hidden_size"]:3d}, L={config["num_layers"]}, '
              f'e={config["embed_dim"]:3d}, lr={config["lr"]:.3f}, '
              f'd={config["dropout"]:.1f}, {bidir_str} '
              f'-> F1={best_val_f1:.4f} ({train_time:.1f}s)')

tuning_df = pd.DataFrame(tuning_results)
print(f'\nTotal trials completed: {len(tuning_df)}')

In [ ]:
# Find best config per model type
best_configs = {}
print('Best configurations per model type (by val F1):')
print('=' * 90)

for model_type in ['RNN', 'LSTM', 'GRU']:
    subset = tuning_df[tuning_df['model_type'] == model_type]
    best_row = subset.loc[subset['best_val_f1'].idxmax()]
    best_configs[model_type] = best_row.to_dict()
    bidir_str = 'Bidirectional' if best_row['bidirectional'] else 'Unidirectional'
    print(f"\n{model_type}:")
    print(f"  hidden_size={int(best_row['hidden_size'])}, num_layers={int(best_row['num_layers'])}, "
          f"embed_dim={int(best_row['embed_dim'])}, lr={best_row['lr']}, "
          f"dropout={best_row['dropout']}, {bidir_str}")
    print(f"  Best val F1: {best_row['best_val_f1']:.4f}, "
          f"Acc: {best_row['best_val_acc']:.4f}, "
          f"Params: {int(best_row['params']):,}")

In [ ]:
# Tuning results visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, model_type in zip(axes, ['RNN', 'LSTM', 'GRU']):
    subset = tuning_df[tuning_df['model_type'] == model_type]
    pivot_data = subset.pivot_table(
        values='best_val_f1',
        index='hidden_size',
        columns='num_layers',
        aggfunc='max'
    )
    sns.heatmap(pivot_data, annot=True, fmt='.4f', cmap='YlGn', ax=ax, vmin=0.8, vmax=1.0)
    ax.set_title(f'{model_type} — Best Val F1 by Hidden Size & Layers')

plt.suptitle('Hyperparameter Tuning Results', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Tuning results scatter — F1 vs Parameters
fig, ax = plt.subplots(figsize=(10, 6))

for model_type in ['RNN', 'LSTM', 'GRU']:
    subset = tuning_df[tuning_df['model_type'] == model_type]
    ax.scatter(
        subset['params'], subset['best_val_f1'],
        color=COLORS[model_type], label=model_type,
        s=80, alpha=0.7, edgecolors='white'
    )

ax.set_xlabel('Number of Parameters', fontsize=12)
ax.set_ylabel('Best Validation F1', fontsize=12)
ax.set_title('Model Complexity vs Performance', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## 7. Final Model Training & Comparison

In [ ]:
# Retrain best models with more epochs
FINAL_EPOCHS = 100
final_models = {}
final_histories = {}
final_times = {}

for model_type in ['RNN', 'LSTM', 'GRU']:
    print(f'\nTraining final {model_type} model...')
    cfg = best_configs[model_type]

    # Reset seed for reproducibility
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    model = TextClassifier(
        model_type=model_type,
        vocab_size=VOCAB_SIZE,
        embed_dim=int(cfg['embed_dim']),
        hidden_size=int(cfg['hidden_size']),
        num_layers=int(cfg['num_layers']),
        dropout=cfg['dropout'],
        bidirectional=cfg['bidirectional']
    )

    history, train_time = train_model(
        model, train_loader, val_loader,
        epochs=FINAL_EPOCHS, lr=cfg['lr'],
        pos_weight=pos_weight, patience=15, verbose=True
    )

    final_models[model_type] = model
    final_histories[model_type] = history
    final_times[model_type] = train_time

    print(f'  {model_type} done: {len(history["val_loss"])} epochs, {train_time:.1f}s')

In [ ]:
# Training curves comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for model_type in ['RNN', 'LSTM', 'GRU']:
    h = final_histories[model_type]
    axes[0, 0].plot(h['train_loss'], label=model_type, color=COLORS[model_type], linewidth=1.5)
    axes[0, 1].plot(h['val_loss'], label=model_type, color=COLORS[model_type], linewidth=1.5)
    axes[1, 0].plot(h['val_acc'], label=model_type, color=COLORS[model_type], linewidth=1.5)
    axes[1, 1].plot(h['val_f1'], label=model_type, color=COLORS[model_type], linewidth=1.5)

axes[0, 0].set_title('Training Loss', fontsize=13)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('BCE Loss')
axes[0, 0].legend(fontsize=11)

axes[0, 1].set_title('Validation Loss', fontsize=13)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('BCE Loss')
axes[0, 1].legend(fontsize=11)

axes[1, 0].set_title('Validation Accuracy', fontsize=13)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].legend(fontsize=11)

axes[1, 1].set_title('Validation F1 Score', fontsize=13)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('F1')
axes[1, 1].legend(fontsize=11)

plt.suptitle('Training Curves: RNN vs LSTM vs GRU', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate all models on TEST set
all_metrics = {}
all_probs = {}
all_labels_dict = {}
all_cms = {}

for model_type in ['RNN', 'LSTM', 'GRU']:
    metrics, probs, labels, cm = evaluate_classifier(
        final_models[model_type], test_loader
    )
    metrics['Parameters'] = final_models[model_type].count_parameters()
    metrics['Training Time'] = f'{final_times[model_type]:.1f}s'

    all_metrics[model_type] = metrics
    all_probs[model_type] = probs
    all_labels_dict[model_type] = labels
    all_cms[model_type] = cm

    print(f'\n{model_type} Test Metrics:')
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f'  {k}: {v:.4f}')
        else:
            print(f'  {k}: {v}')

In [ ]:
# Confusion matrices side-by-side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, model_type in zip(axes, ['RNN', 'LSTM', 'GRU']):
    cm = all_cms[model_type]
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'],
        annot_kws={'fontsize': 14}
    )
    ax.set_title(f'{model_type} Confusion Matrix', fontsize=13)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

    # Add FP/FN annotations
    fp = cm[0, 1]
    fn = cm[1, 0]
    ax.text(0.5, -0.15, f'FP={fp} (ham->spam)  FN={fn} (spam->ham)',
            transform=ax.transAxes, ha='center', fontsize=10, style='italic')

plt.suptitle('Confusion Matrices on Test Set', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves overlay
fig, ax = plt.subplots(figsize=(8, 8))

for model_type in ['RNN', 'LSTM', 'GRU']:
    fpr, tpr, _ = roc_curve(all_labels_dict[model_type], all_probs[model_type])
    auc_val = all_metrics[model_type]['AUC-ROC']
    ax.plot(fpr, tpr, color=COLORS[model_type], linewidth=2,
            label=f'{model_type} (AUC = {auc_val:.4f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random (AUC = 0.5)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves: RNN vs LSTM vs GRU', fontsize=14)
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])
plt.tight_layout()
plt.show()

In [ ]:
# Precision-Recall curves overlay
fig, ax = plt.subplots(figsize=(8, 8))

for model_type in ['RNN', 'LSTM', 'GRU']:
    precision_vals, recall_vals, _ = precision_recall_curve(
        all_labels_dict[model_type], all_probs[model_type]
    )
    ap = average_precision_score(all_labels_dict[model_type], all_probs[model_type])
    ax.plot(recall_vals, precision_vals, color=COLORS[model_type], linewidth=2,
            label=f'{model_type} (AP = {ap:.4f})')

# Baseline: proportion of positive class
baseline = y_test.mean()
ax.axhline(baseline, color='gray', linestyle='--', alpha=0.5, label=f'Baseline ({baseline:.2f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves: RNN vs LSTM vs GRU', fontsize=14)
ax.legend(loc='lower left', fontsize=11)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])
plt.tight_layout()
plt.show()

In [ ]:
# Classification reports
for model_type in ['RNN', 'LSTM', 'GRU']:
    preds = (all_probs[model_type] >= 0.5).astype(int)
    print(f'\n{"="*60}')
    print(f'{model_type} Classification Report')
    print(f'{"="*60}')
    print(classification_report(
        all_labels_dict[model_type], preds,
        target_names=['Ham', 'Spam'], digits=4
    ))

In [ ]:
# TF-IDF + Logistic Regression baseline
print('Training TF-IDF + Logistic Regression baseline...')

# Use raw text (not tokenized) for TF-IDF
X_train_text = df.iloc[train_test_split(
    np.arange(len(df)), test_size=0.30, random_state=SEED, stratify=df['label_enc']
)[0]]['message'].values

# Simpler approach: re-split with same seed
texts = df['message'].values
labels_all = df['label_enc'].values

txt_train, txt_temp, lbl_train, lbl_temp = train_test_split(
    texts, labels_all, test_size=0.30, random_state=SEED, stratify=labels_all
)
txt_val, txt_test, lbl_val, lbl_test = train_test_split(
    txt_temp, lbl_temp, test_size=0.50, random_state=SEED, stratify=lbl_temp
)

# Build pipeline
tfidf_lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english')),
    ('lr', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED))
])

start_time = time.time()
tfidf_lr_pipeline.fit(txt_train, lbl_train)
baseline_time = time.time() - start_time

# Evaluate on test set
baseline_probs = tfidf_lr_pipeline.predict_proba(txt_test)[:, 1]
baseline_preds = tfidf_lr_pipeline.predict(txt_test)

baseline_metrics = {
    'Accuracy': accuracy_score(lbl_test, baseline_preds),
    'F1': f1_score(lbl_test, baseline_preds),
    'Precision': precision_score(lbl_test, baseline_preds),
    'Recall': recall_score(lbl_test, baseline_preds),
    'AUC-ROC': roc_auc_score(lbl_test, baseline_probs),
    'Parameters': sum(tfidf_lr_pipeline.named_steps['lr'].coef_.size) + 1,
    'Training Time': f'{baseline_time:.1f}s'
}

all_metrics['TF-IDF+LR'] = baseline_metrics

print(f'\nTF-IDF + Logistic Regression Baseline:')
for k, v in baseline_metrics.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

print(f'\n{classification_report(lbl_test, baseline_preds, target_names=["Ham", "Spam"], digits=4)}')

In [ ]:
# Final comparison table
print('\n' + '=' * 90)
print('FINAL COMPARISON TABLE — Test Set Results')
print('=' * 90)

metrics_df = pd.DataFrame(all_metrics).T
for col in ['Accuracy', 'F1', 'Precision', 'Recall', 'AUC-ROC']:
    metrics_df[col] = metrics_df[col].apply(
        lambda x: f'{x:.4f}' if isinstance(x, float) else x
    )

metrics_df

In [ ]:
# Final summary visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models_list = ['RNN', 'LSTM', 'GRU', 'TF-IDF+LR']
colors_bar = [COLORS['RNN'], COLORS['LSTM'], COLORS['GRU'], '#95a5a6']

# 1. F1 comparison
f1_vals = [float(all_metrics[m]['F1']) for m in models_list]
bars = axes[0].bar(models_list, f1_vals, color=colors_bar, edgecolor='white', width=0.5)
for bar, val in zip(bars, f1_vals):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', fontweight='bold', fontsize=10)
axes[0].set_title('F1 Score Comparison', fontsize=13)
axes[0].set_ylabel('F1 Score')
axes[0].set_ylim([0.5, 1.05])

# 2. AUC-ROC comparison
auc_vals = [float(all_metrics[m]['AUC-ROC']) for m in models_list]
bars = axes[1].bar(models_list, auc_vals, color=colors_bar, edgecolor='white', width=0.5)
for bar, val in zip(bars, auc_vals):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', fontweight='bold', fontsize=10)
axes[1].set_title('AUC-ROC Comparison', fontsize=13)
axes[1].set_ylabel('AUC-ROC')
axes[1].set_ylim([0.5, 1.05])

# 3. Parameters (RNN models only)
rnn_models = ['RNN', 'LSTM', 'GRU']
param_vals = [all_metrics[m]['Parameters'] for m in rnn_models]
bars = axes[2].bar(rnn_models, param_vals,
                   color=[COLORS[m] for m in rnn_models], edgecolor='white', width=0.5)
for bar, val in zip(bars, param_vals):
    axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
                 f'{val:,}', ha='center', fontweight='bold', fontsize=10)
axes[2].set_title('Parameter Count (RNN Models)', fontsize=13)
axes[2].set_ylabel('Parameters')

plt.suptitle('Final Summary: RNN vs LSTM vs GRU vs TF-IDF+LR on SMS Spam', fontsize=14)
plt.tight_layout()
plt.show()

---
## 8. Analysis & Insights

In [ ]:
# Misclassified examples analysis
print('='*80)
print('MISCLASSIFICATION ANALYSIS')
print('='*80)

# Use LSTM model for detailed analysis (or best model)
analysis_model_type = 'LSTM'
analysis_probs = all_probs[analysis_model_type]
analysis_labels = all_labels_dict[analysis_model_type]
analysis_preds = (analysis_probs >= 0.5).astype(int)

# Get test set messages
_, temp_idx = train_test_split(
    np.arange(len(df)), test_size=0.30, random_state=SEED, stratify=df['label_enc']
)
_, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=SEED, stratify=df['label_enc'].values[temp_idx]
)
test_messages = df.iloc[test_idx]['message'].values
test_true_labels = df.iloc[test_idx]['label'].values

# False Positives: Ham classified as Spam
fp_mask = (analysis_labels == 0) & (analysis_preds == 1)
fp_indices = np.where(fp_mask)[0]
print(f'\nFalse Positives ({len(fp_indices)} total) — Ham misclassified as Spam:')
for i, idx in enumerate(fp_indices[:5]):
    msg = test_messages[idx]
    prob = analysis_probs[idx]
    display_msg = msg[:100] + '...' if len(msg) > 100 else msg
    print(f'  [{i+1}] P(spam)={prob:.3f} | {display_msg}')

# False Negatives: Spam classified as Ham
fn_mask = (analysis_labels == 1) & (analysis_preds == 0)
fn_indices = np.where(fn_mask)[0]
print(f'\nFalse Negatives ({len(fn_indices)} total) — Spam misclassified as Ham:')
for i, idx in enumerate(fn_indices[:5]):
    msg = test_messages[idx]
    prob = analysis_probs[idx]
    display_msg = msg[:100] + '...' if len(msg) > 100 else msg
    print(f'  [{i+1}] P(spam)={prob:.3f} | {display_msg}')

print(f'\nTotal misclassifications: {(analysis_preds != analysis_labels).sum()} '
      f'out of {len(analysis_labels)} ({(analysis_preds != analysis_labels).mean()*100:.1f}%)')

In [ ]:
# Bidirectional vs Unidirectional comparison
print('Bidirectional vs Unidirectional Comparison')
print('='*60)

bidir_results = []

for model_type in ['RNN', 'LSTM', 'GRU']:
    for bidir in [False, True]:
        torch.manual_seed(SEED)
        np.random.seed(SEED)

        model = TextClassifier(
            model_type=model_type, vocab_size=VOCAB_SIZE,
            embed_dim=100, hidden_size=64, num_layers=1,
            dropout=0.2, bidirectional=bidir
        )

        history, train_time = train_model(
            model, train_loader, val_loader,
            epochs=30, lr=0.001, pos_weight=pos_weight, verbose=False
        )

        metrics, _, _, _ = evaluate_classifier(model, test_loader)
        bidir_results.append({
            'Model': model_type,
            'Bidirectional': bidir,
            'Direction': 'Bidirectional' if bidir else 'Unidirectional',
            'Test F1': metrics['F1'],
            'Test Acc': metrics['Accuracy'],
            'Test AUC': metrics['AUC-ROC'],
            'Params': model.count_parameters(),
            'Time': train_time
        })

        bidir_str = 'Bi' if bidir else 'Uni'
        print(f'  {model_type} {bidir_str:3s}: F1={metrics["F1"]:.4f}, '
              f'AUC={metrics["AUC-ROC"]:.4f}, '
              f'Params={model.count_parameters():,}, Time={train_time:.1f}s')

bidir_df = pd.DataFrame(bidir_results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 comparison
for i, model_type in enumerate(['RNN', 'LSTM', 'GRU']):
    subset = bidir_df[bidir_df['Model'] == model_type]
    x = np.arange(2)
    axes[0].bar(
        x + i * 0.25, subset['Test F1'].values,
        width=0.25, label=model_type, color=COLORS[model_type], edgecolor='white'
    )

axes[0].set_xticks([0.25, 1.25])
axes[0].set_xticklabels(['Unidirectional', 'Bidirectional'])
axes[0].set_ylabel('Test F1')
axes[0].set_title('F1: Uni vs Bi-directional', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].set_ylim([0.5, 1.05])

# Parameter comparison
for i, model_type in enumerate(['RNN', 'LSTM', 'GRU']):
    subset = bidir_df[bidir_df['Model'] == model_type]
    x = np.arange(2)
    axes[1].bar(
        x + i * 0.25, subset['Params'].values,
        width=0.25, label=model_type, color=COLORS[model_type], edgecolor='white'
    )

axes[1].set_xticks([0.25, 1.25])
axes[1].set_xticklabels(['Unidirectional', 'Bidirectional'])
axes[1].set_ylabel('Parameters')
axes[1].set_title('Parameters: Uni vs Bi-directional', fontsize=13)
axes[1].legend(fontsize=10)

plt.suptitle('Bidirectional vs Unidirectional Comparison', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Embedding dimension ablation
print('Embedding Dimension Ablation Study')
print('='*60)

embed_dims = [25, 50, 100, 150, 200]
embed_results = []

for embed_dim in embed_dims:
    for model_type in ['RNN', 'LSTM', 'GRU']:
        torch.manual_seed(SEED)
        np.random.seed(SEED)

        model = TextClassifier(
            model_type=model_type, vocab_size=VOCAB_SIZE,
            embed_dim=embed_dim, hidden_size=64, num_layers=1,
            dropout=0.2, bidirectional=False
        )

        history, _ = train_model(
            model, train_loader, val_loader,
            epochs=30, lr=0.001, pos_weight=pos_weight, verbose=False
        )

        metrics, _, _, _ = evaluate_classifier(model, test_loader)
        embed_results.append({
            'embed_dim': embed_dim, 'model_type': model_type,
            'test_f1': metrics['F1'], 'params': model.count_parameters()
        })

    print(f'  embed_dim={embed_dim:3d} done')

embed_df = pd.DataFrame(embed_results)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for model_type in ['RNN', 'LSTM', 'GRU']:
    subset = embed_df[embed_df['model_type'] == model_type]
    ax1.plot(subset['embed_dim'], subset['test_f1'], 'o-',
             color=COLORS[model_type], label=model_type, linewidth=2, markersize=8)
    ax2.plot(subset['embed_dim'], subset['params'], 'o-',
             color=COLORS[model_type], label=model_type, linewidth=2, markersize=8)

ax1.set_xlabel('Embedding Dimension', fontsize=12)
ax1.set_ylabel('Test F1 Score', fontsize=12)
ax1.set_title('F1 vs Embedding Dimension', fontsize=13)
ax1.legend(fontsize=11)
ax1.set_xticks(embed_dims)

ax2.set_xlabel('Embedding Dimension', fontsize=12)
ax2.set_ylabel('Total Parameters', fontsize=12)
ax2.set_title('Parameters vs Embedding Dimension', fontsize=13)
ax2.legend(fontsize=11)
ax2.set_xticks(embed_dims)

plt.suptitle('Effect of Embedding Dimension', fontsize=14)
plt.tight_layout()
plt.show()

print('Observations:')
print('- Larger embeddings increase parameters linearly (dominated by vocab_size * embed_dim)')
print('- Diminishing returns beyond embed_dim=100 for this dataset')
print('- Smaller embeddings (25-50) may underfit the vocabulary semantics')

### Key Insights

1. **Class imbalance matters**: Using `pos_weight` in `BCEWithLogitsLoss` is critical. Without it, models would achieve ~87% accuracy by simply predicting all messages as ham, but miss most spam.

2. **F1 > Accuracy for imbalanced data**: Accuracy is misleading here. A model predicting all-ham gets 87% accuracy but 0% spam recall. F1 score, AUC-ROC, and the confusion matrix are far more informative.

3. **LSTM/GRU vs RNN**: The gated architectures (LSTM, GRU) generally outperform vanilla RNN for text classification because:
   - SMS messages can have important context words spread throughout the message
   - RNN suffers from vanishing gradients on longer sequences
   - Gates help selectively remember/forget information across the sequence

4. **GRU as a practical choice**: GRU often matches or approaches LSTM performance with fewer parameters and faster training. For SMS-length texts, the simpler gating of GRU is often sufficient.

5. **Bidirectional processing**: Can help for classification tasks since the full message is available at inference time (unlike autoregressive generation). Forward and backward context together give a richer representation.

6. **TF-IDF + LR baseline is strong**: Traditional ML baselines remain competitive on short-text classification tasks. The recurrent models' advantage grows with:
   - Longer sequences where word order matters more
   - More training data to learn embeddings effectively
   - Tasks requiring understanding of word context and relationships

7. **Embedding dimension tradeoff**: Larger embeddings capture richer word semantics but increase parameters proportional to vocabulary size. For a ~4K vocabulary, 50-100 dimensions is usually sufficient.

---
## 9. Conclusion

### Final Results Summary

| Model | Accuracy | F1 | AUC-ROC | Precision | Recall | Parameters | Training Time |
|-------|----------|-----|---------|-----------|--------|------------|---------------|
| RNN | -- | -- | -- | -- | -- | -- | -- |
| LSTM | -- | -- | -- | -- | -- | -- | -- |
| GRU | -- | -- | -- | -- | -- | -- | -- |
| TF-IDF+LR | -- | -- | -- | -- | -- | -- | -- |

*(Values filled in dynamically above in Section 7)*

### Key Findings

1. **All three recurrent models achieve strong spam detection** on this dataset, with F1 scores typically above 0.90.
2. **GRU provides the best efficiency-performance tradeoff** — fewer parameters than LSTM with comparable F1.
3. **LSTM often achieves the highest absolute F1/AUC** thanks to its more expressive gating mechanism.
4. **Vanilla RNN is competitive** on short texts (SMS are typically <50 words), where long-range dependencies are less critical.
5. **TF-IDF + Logistic Regression is a strong baseline** — always compare neural models against it!

### RNN vs LSTM vs GRU for Text Classification

| Aspect | RNN | LSTM | GRU |
|--------|-----|------|-----|
| **Parameters** | Fewest | Most (~4x RNN) | Middle (~3x RNN) |
| **Vanishing Gradient** | Severe | Mitigated by gates | Mitigated by gates |
| **Short Text (SMS)** | Competitive | Best | Near-best |
| **Long Text (documents)** | Poor | Best | Good |
| **Training Speed** | Fastest/epoch | Slowest/epoch | Middle |
| **Best Use Case** | Short sequences, speed-critical | Long sequences, accuracy-critical | General-purpose default |

### Extensions
- **Pre-trained embeddings**: Use GloVe or Word2Vec instead of learning from scratch
- **Attention mechanism**: Add attention over RNN hidden states for interpretability
- **Character-level models**: RNNs over characters to capture subword patterns (useful for SMS abbreviations)
- **Transformer models**: Compare with BERT or DistilBERT fine-tuning
- **Multi-class classification**: Extend to multi-label or multi-class text tasks

In [ ]:
print('Notebook complete!')
print('='*60)
print('Case Study 4: Binary Text Classification — Spam SMS')
print('RNN vs LSTM vs GRU Comparison')
print('='*60)
print(f'\nDataset: SMS Spam Collection ({len(df):,} messages)')
print(f'Vocabulary size: {VOCAB_SIZE:,}')
print(f'Max sequence length: {MAX_SEQ_LEN}')
print(f'\nModels trained and evaluated on test set.')
print(f'See Section 7 for detailed comparison table and visualizations.')
print(f'\nProceeding to next case study...')